In [1]:
import requests
from bs4 import BeautifulSoup

news_rss = requests.get('https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07')
news_rss_soup = BeautifulSoup(news_rss.content, 'xml')
link_list = news_rss_soup.select('item > link')
print("기사 개수:", len(link_list))

기사 개수: 29


In [2]:
title_list = news_rss_soup.select('item > title')
title_list = [title.text for title in title_list]
print(title_list)

["[자막뉴스] 사무직의 '블루칼라 러시' 시작되나…'임금 역전' 현상 확산에 앞다퉈서", '"트럼프, 자기 과시·망상·현실 외면"…국정연설 비판한 민주당', '[오그랲] 2030 지지율 87%…극우에 열광하는 일본 MZ?', '트럼프 \'황금 시대\' 주장 반박한 민주당…"생활비 부담에 고통"', '"거짓·근거 부족"…미 언론들, 트럼프 연설 실시간 팩트 체크', '[현장영상] "일어나 박수 안쳐? 부끄러운 줄 알아"…\'이민자 추방\' 자랑에 발 구르며 \'환호\'', "'AI 안전' 강조한 앤트로픽 '후퇴'…이유는 '규제 부재'", '[자막뉴스] "자율 살상 무기에만은 제발"…"이틀 줄게 권한 넘겨" 최후통첩', "일 '로봇 승려' 시대 오나…경전 읊고 합장하는 '붓다로이드' 등장", '108분 쩌렁쩌렁 연설한 트럼프 "관세 더 세진다"…이란엔 경고', '트럼프, 국정연설서 한·북·중 등 동북아 언급 전무…집권 1기와 대비', "'AI 관련주 견인' 일 닛케이지수 58,000선 재돌파…장중 최고가 경신", '트럼프, 미 최장 국정연설…민주당엔 "미쳤다"·하키팀엔 \'활짝\' 극과 극', '"수작업 코딩 시대 끝났다"…호주 기업, 인력 30% 감축', '트럼프 "이란과 외교 선호하지만 아직 핵무기 포기 의사 못 들어"', '트럼프 "기업들, 자체 발전소로 AI 데이터 센터 전력 수요 해결해야"', '트럼프 "다수 국가, 무역 합의 유지 원해…그들에 더 나쁜 합의도 가능"', '14억 원 걸고 "어머니 찾아달라"…대통령도 나섰지만', '트럼프 집권 2기 첫 국정연설…"관세 정책 그대로"', "빗길·산악도 거뜬한 고성능 '강아지 로봇'…중국 유니트리 공개", '[바로이뉴스] "USA! USA!" 본격 \'미국 부흥회\'…"관세 위법" 대법원장 앞에선', "선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [취재파일]", '트럼프 "지금이 미 황금시대…우리의 적들은 두려워하고 있다"', '군인 화장했더니 유골서 숟가락…"이게 뭐냐" 태국 발칵', '중국 드론 

In [10]:
news_data = []
for link in link_list:
    news_response = requests.get(link.text)
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser')
    news_content = news_content_soup.select_one("#container > div.w_inner > div.w_article > div.w_article_cont > div.w_article_left > div.article_cont_area > div.main_text > div")
    news_data.append(news_content.text)

import pandas as pd
news_df = pd.DataFrame(data={'title': title_list, 'content': news_data})
print(news_df.head())

news_df.to_csv("news.csv", encoding="utf-8-sig", index=False)
print("Save complete")

                                              title  \
0  [바로이뉴스] "USA! USA!" 본격 '미국 부흥회'…"관세 위법" 대법원장 앞에선   
1               선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [취재파일]   
2                         [속보] 트럼프, 집권 2기 첫 국정연설 시작   
3                    군인 화장했더니 유골서 숟가락…"이게 뭐냐" 태국 발칵   
4              중국 드론 업체 DJI "증거 없이 수입 금지"…미국 법원에 소송   

                                             content  
0  \n  현지시간 24일 미 워싱턴 국회의사당 하원 회의장에서 트럼프 미 대통령의 집...  
1  \n\n관세 위법, 엡스타인 파동…진퇴양난 트럼프\n 도널드 트럼프 미 대통령 요즘...  
2  \n\n▲ 국정연설하는 트럼프 미 대통령\n\n 트럼프, 집권 2기 첫 국정연설 시...  
3  \n  태국에서 군 복무 중 숨진 군인의 유골에서 숟가락이 발견되는 일이 벌어져 태...  
4  \n\n▲ 미국 상점에 전시된 DJI 무인기\n\n 외국산 신형 무인기(드론) 및 ...  
Save complete


# 뉴스 컨텐츠 클린징

In [5]:
news_data_1 = []
for link in link_list:
    news_response = requests.get(link.text)
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser')

    news_content = news_content_soup.select_one("div[itemprop=articleBody]")

    if news_content is not None:
          news_data_1.append(news_content.text.strip())
    else:
        news_data_1.append("본문 없음")  # 또는 None, ""
news_data_1_df = pd.DataFrame(data={'title': title_list, 'content': news_data_1})

In [6]:
news_data_1_df.head()

,title,content
0,"미국, 인도·인도네시아 태양광 고율 관세…한화, 반사 이익?",▲ 태양광 에너지\n\n 미국 정부가 인도와 인도네시아 등지에서 생산되는 태양광 발...
1,미국 최대 은행 JP모건도 AI발 대규모 인력 재배치 계획,▲ 제이미 다이먼 JP모건 회장\n\n 미국 최대 은행인 JP모건체이스의 제이미 다...
2,"영국, 입국 전 전자여행허가 확대…한국 포함 85개국에 의무화","▲ 영국의 한 공항\n\n 영국이 25일(현지시간)부터 무비자 입국이 가능한 한국,..."
3,'상호관세 돌려달라' 소송 봇물…로레알·다이슨도 합류,▲ 미국 연방 대법원\n\n 미국 연방 대법원이 최근 위법으로 판결한 도널드 트럼프...
4,"멕시코 마약왕 사살 혼란에…혼다, 현지 공장 가동중단 뒤 재개",▲ 일본 자동차 브랜드 혼다\n\n 멕시코 당국이 마약 카르텔 두목을 사살한 뒤 폭...


In [7]:

missing_count = sum(1 for content in news_data_1 if content == "본문 없음")
print(f"본문이 없는 기사 개수: {missing_count}개")
print(f"전체 기사 개수: {len(news_data_1)}개")

본문이 없는 기사 개수: 0개
전체 기사 개수: 29개


In [8]:
import pandas as pd

news_df = pd.DataFrame({
    'title': title_list,
    'content': news_data
})
news_df['is_missing'] = news_df['content'] == "본문 없음"
missing_count = news_df['is_missing'].sum()
total_count = len(news_df)

print(f"본문 없음: {missing_count}개 / 전체: {total_count}개 ({missing_count/total_count:.1%})")

본문 없음: 0개 / 전체: 29개 (0.0%)
